# A. Importation of libraries and Configs

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
class Config:
    # Path to the pseudonimized revenues dataset
    dataset_dir = r"Database\revenues_pseudonymized.xlsx"
    # Path to the enrollee infos
    enrollees_dir = r"Database\enrollees_pseudonymized.xlsx"
    # Path to the machine learning model parameters
    parameters_dir = r"machine_learning\parameters.json"

    # Path to cache directory to store preprocessed dataset if needed
    cache_dir = ""
    load_cache = True

    # Path to store transformer results
    results_dir = r"C:\Users\rjbel\Python\Data\Thesis\Results"


    # The date used
    observation_end = datetime.today()

    # Class to predict
    target_feature = 'dtp_bracket'
    # Test size in %
    test_size = 0.3

    # Time points used in generating survival features
    # It's not until 120 since the earliest pre-payment is 288 days
    time_points = [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360, 390, 420, 450]


args = Config()

# B. Loading of datasets

## 1. Revenues

In [ ]:
df_revenues = pd.read_excel(args.dataset_dir)

In [ ]:
df_revenues

## 2. Enrollees

In [ ]:
df_enrollees = pd.read_excel(args.enrollees_dir)

In [ ]:
df_enrollees

## 3. Credit Sales

In [ ]:
from feature_engineering.credit_sales_machine_learning import CreditSales

cs = CreditSales(df_revenues, df_enrollees, args)
df_credit_sales = cs.show_data()

In [ ]:
df_credit_sales

# C. Exploratory Data Analysis

In [ ]:
# Get counts
counts = df_credit_sales.dtp_bracket.value_counts()

# Convert to percentages
percentages = counts / counts.sum() * 100

# Combine into one DataFrame
result = pd.DataFrame({
    'count': counts,
    'percentage': percentages.round(2)
})

print(result)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Remove those that have no full dtp_history:
df_filtered = df_credit_sales.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'])


# Clean and filter directly on the DataFrame
df_filtered = df_filtered.loc[
    df_filtered['days_elapsed_until_fully_paid']
        .replace("", np.nan)   # replace empty strings with NaN
        .dropna()              # drop NaNs
        .index                 # keep aligned index
]


# Apply numeric filter
df_filtered = df_filtered[
    (df_filtered['days_elapsed_until_fully_paid'] >= -200) &
    (df_filtered['days_elapsed_until_fully_paid'] <= 200)
]

# Convert censor column to categorical with labels
df_filtered["censor_label"] = (
    df_filtered["censor"]
    .map({0: "Still Unpaid", 1: "Fully Paid"})
    .astype("category")   # force categorical type
)


# KDE plot with grouping by categorical censor labels
sns.kdeplot(
    data=df_filtered,
    x="days_elapsed_until_fully_paid",
    hue="censor_label",
    fill=False,
    common_norm=False,
    palette="Set1"
)

plt.title("KDE Plot: Days Elapsed Until Fully Paid (-200 to +200)")
plt.xlabel("Days Elapsed")
plt.ylabel("Density")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_filtered = df_credit_sales[df_credit_sales['censor'] == 1]
df_filtered = df_filtered.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'])

# Select relevant columns
cols = ['days_elapsed_until_fully_paid', 
        'dtp_1', 'dtp_2', 'dtp_3', 'dtp_4', 
        'dtp_avg', 'dtp_wavg', 'dtp_2_trend',
        'dtp_3_trend', 'days_since_last_payment',
        'credit_sale_amount', 'amount_due_cumsum',
        'amount_paid_cumsum', 'opening_balance']

# Compute correlation matrix
corr = df_filtered[cols].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Correlation with Days Elapsed Until Fully Paid")
plt.show()

# D. Machine Learning Pipelines

## 1. Survival Analysis Sandbox

### a. Step by step pipeline

In [ ]:
# Helper columns
drop_columns = ['school_year', 'student_id_pseudonimized', 'category_name',
       'gross_receivables', 'amount_discounted', 'adjustments', 'due_date_prev_1',
       'due_date_prev_2', 'date_fully_paid', 'last_payment_date']

# Plan related columns
drop_columns = drop_columns + ['plan_type_Plan - A', 'plan_type_Plan - B', 'plan_type_Plan - C',
                               'plan_type_Plan - D', 'plan_type_Plan - E', 'plan_type_nan']

df_data = df_credit_sales[df_credit_sales['censor'] == 1]

df_data = df_data.drop(columns=drop_columns)
df_data.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'], inplace=True)
df_data

In [ ]:
drop_columns = ['school_year', 'student_id_pseudonimized', 'category_name',
       'gross_receivables', 'amount_discounted', 'adjustments', 'due_date',
       'due_date_prev_1', 'due_date_prev_2', 'date_fully_paid', 'last_payment_date',
       'plan_type_Plan - D', 'plan_type_Plan - E', 'plan_type_nan', 'dtp_bracket']

drop_columns = drop_columns + ['plan_type_Plan - A', 'plan_type_Plan - B', 'plan_type_Plan - C']

df_data_surv = df_credit_sales.drop(columns=drop_columns)
df_data_surv.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'], inplace=True)
df_data_surv

In [ ]:
df_surv = df_data_surv.copy()

X = df_surv.drop(columns=['days_elapsed_until_fully_paid', 'censor'])
T = df_surv['days_elapsed_until_fully_paid']
E = df_surv['censor']

# Avoid negative values by shifting by the days of pre-paid period
earliest_payment = np.minimum(T, 0) # Maximum to only get pre-payments
ε = 1e-6 # Used to avoid zero values
T = T - earliest_payment + ε

In [ ]:
earliest_payment

In [ ]:
import xgboost as xgb
print(xgb.build_info())  # Should show cuda_version if GPU-enabled

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

# ============================================
# STEP X — Plot Kaplan-Meier Survival Curve
# ============================================

# Initialize Kaplan-Meier fitter
kmf = KaplanMeierFitter()

# Fit the model on your training data
kmf.fit(durations=T,
        event_observed=E,
        label="Kaplan-Meier Curve")

# Plot the survival function
plt.figure(figsize=(8,6))
kmf.plot_survival_function()
plt.title("Kaplan-Meier Survival Curve for Invoice Payments")
plt.xlabel("Days to Payment")
plt.ylabel("Survival Probability")
plt.xlim(-3, 100)
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from lifelines import CoxPHFitter

# ============================================
# STEP 0 — Prepare Survival Data
# ============================================
survival_train = np.array(
    [(bool(e), float(t)) for e, t in zip(E.values, T.values)],
    dtype=[('event', 'bool'), ('time', 'float')]
)

# ============================================
# STEP 1 — Preprocess Predictors
# ============================================
# Drop highly correlated features (>0.95)
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
X_reduced = X.drop(columns=to_drop)

# Standardize predictors
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_reduced), columns=X_reduced.columns)

# ============================================
# STEP 2 — Hyperparameter Grids
# ============================================
coxnet_params = {
    "l1_ratio": [0.2, 0.5, 0.8, 1.0],
    "alphas": [[1e-6], [1e-5], [1e-4], [1e-3], [1e-2]]
}

coxph_params = {
    "penalizer": [0.01, 0.1, 1, 10, 100],
    "baseline_estimation_method": ["breslow"]  # ✅ only use breslow for stability
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ============================================
# STEP 3 — CoxnetSurvivalAnalysis Tuning
# ============================================
best_c_index, best_params = -np.inf, None

for l1 in coxnet_params["l1_ratio"]:
    for alpha in coxnet_params["alphas"]:
        c_indices = []
        for train_idx, val_idx in kf.split(X_scaled):
            X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
            y_train, y_val = survival_train[train_idx], survival_train[val_idx]

            model = CoxnetSurvivalAnalysis(l1_ratio=l1, alphas=alpha, max_iter=10000, tol=1e-7)

            # ✅ Suppress "all coefficients are zero" warning
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message="all coefficients are zero, consider decreasing alpha",
                    category=UserWarning
                )
                model.fit(X_train, y_train)

            risk_scores = model.predict(X_val)
            c_index = concordance_index_censored(
                y_val["event"], y_val["time"], risk_scores
            )[0]
            c_indices.append(c_index)

        mean_c_index = np.mean(c_indices)
        if mean_c_index > best_c_index:
            best_c_index = mean_c_index
            best_params = {"l1_ratio": l1, "alphas": alpha}

print("Best Coxnet Params:", best_params, "Best C-index:", best_c_index)

# ============================================
# STEP 4 — CoxPHFitter Tuning
# ============================================
best_c_index_ph, best_params_ph = -np.inf, None

for penalizer in coxph_params["penalizer"]:
    for method in coxph_params["baseline_estimation_method"]:
        c_indices = []
        for train_idx, val_idx in kf.split(X_scaled):
            X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
            y_train, y_val = survival_train[train_idx], survival_train[val_idx]

            df_train = X_train.copy()
            df_train["time"] = y_train["time"]
            df_train["event"] = y_train["event"]

            df_val = X_val.copy()
            df_val["time"] = y_val["time"]
            df_val["event"] = y_val["event"]

            try:
                cph = CoxPHFitter(penalizer=penalizer, baseline_estimation_method=method)
                cph.fit(df_train, duration_col="time", event_col="event", robust=True)

                risk_scores = cph.predict_partial_hazard(df_val)
                c_index = concordance_index_censored(
                    y_val["event"], y_val["time"], risk_scores.values
                )[0]
                c_indices.append(c_index)

            except Exception as e:
                print(f"Skipped penalizer={penalizer}, method={method} due to error: {e}")
                continue

        if c_indices:  # only update if successful
            mean_c_index = np.mean(c_indices)
            if mean_c_index > best_c_index_ph:
                best_c_index_ph = mean_c_index
                best_params_ph = {"penalizer": penalizer, "baseline_estimation_method": method}

print("Best CoxPHFitter Params:", best_params_ph, "Best C-index:", best_c_index_ph)

In [ ]:
from machine_learning.utils.data.data_preparation import DataPreparer

preparer = DataPreparer(
    df_data,
    target_feature=args.target_feature,
    test_size=args.test_size
)
preparer.prep_data()

X_train = preparer.X_train
X_test = preparer.X_test
y_train = preparer.y_train
y_test = preparer.y_test

In [ ]:
from machine_learning.utils.features.generate_survival_features import generate_survival_features

best_surv_params = {
    'penalizer': 0.1,
    'l1_ratio': 1,
    'baseline_estimation_method': 'breslow',
    'robust': True,
    'step_size': 0.5
  }

df_survival_train, df_survival_test = generate_survival_features(df_data_surv, T, E, X_train, X_test, best_surv_params, time_points=args.time_points)

In [ ]:
model_parameters = {
    "learning_rate": 0.1,
    "n_estimators": 50,
    "random_state": 42
}

In [ ]:
from machine_learning.utils.data.data_preparation import DataPreparer 
from machine_learning import AdaBoostPipeline

preparer = DataPreparer(df_data, args.target_feature, test_size=args.test_size, verbose=False)
preparer.prep_data(balance_strategy="borderline_smote")

X_train, X_test = preparer.X_train, preparer.X_test
y_train, y_test = preparer.y_train, preparer.y_test

pipeline = AdaBoostPipeline(
    X_train, X_test, y_train, y_test,
    args,
    model_parameters
)

# Capture results from pipeline
result = pipeline.initialize_model().fit(use_feature_selection=True).evaluate().show_results()
result

In [ ]:
from machine_learning.utils.data.data_preparation import DataPreparer
from machine_learning import AdaBoostPipeline

pipeline = AdaBoostPipeline(
    df_survival_train, df_survival_test, y_train, y_test,
    args,
    model_parameters
)

# Capture results from pipeline
result = pipeline.initialize_model().fit(use_feature_selection=True).evaluate().show_results()
result

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import AdaBoostClassifier
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

from machine_learning.utils.features.generate_survival_features import generate_survival_features

# ============================================
# STEP 0 — Initialize and Prepare Data
# ============================================

# Build survival array
survival_train = np.array(
    [(bool(e), float(t)) for e, t in zip(E.values, T.values)],
    dtype=[('event', 'bool'), ('time', 'float')]
)

# ============================================
# STEP 1 — Tune Survival Model
# ============================================
penalties = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]
best_penalty, best_c_index = None, -np.inf

for λ in penalties:
    model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[λ])
    model.fit(X, survival_train)
    risk_scores = model.predict(X)
    c_index = concordance_index_censored(E.astype(bool), T, risk_scores)[0]
    if c_index > best_c_index:
        best_penalty, best_c_index = λ, c_index

print("Best Penalty:", best_penalty, "Best C-index:", best_c_index)

# ============================================
# STEP 2 — Retrain Survival Model
# ============================================
survival_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[best_penalty])
survival_model.fit(df_data_surv, survival_train)

# ============================================
# STEP 3 — Generate Survival Features
# ============================================
df_survival_train, df_survival_test = generate_survival_features(X, T, E, X_train, X_test, best_surv_params, time_points=args.time_points)

# ============================================
# STEP 4 — Train Classifier (baseline vs survival)
# ============================================
# Baseline classifier (raw features, full dataset)
baseline_clf = AdaBoostClassifier(**model_parameters)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_f1 = f1_score(y_test, baseline_pred, average="macro")

# Survival-feature classifier (restricted subset)
clf = AdaBoostClassifier(**model_parameters)
clf.fit(df_survival_train, y_train)
survival_pred = clf.predict(df_survival_test)
survival_f1 = f1_score(y_test, survival_pred, average="macro")

print("Baseline F1:", baseline_f1)
print("Survival F1:", survival_f1)

# ============================================
# STEP 5 — Feature Selection + Final Classifier
# ============================================
selector = SelectFromModel(clf, prefit=True)
X_train_selected = selector.transform(df_survival_train)
X_test_selected  = selector.transform(df_survival_test)

final_clf = AdaBoostClassifier(**model_parameters)
final_clf.fit(X_train_selected, y_train)
final_pred = final_clf.predict(X_test_selected)
final_f1 = f1_score(y_test, final_pred, average="macro")

print("Final F1 Score:", final_f1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Count frequency of each unique duration
freq = T.value_counts().sort_index()

# Plot
plt.figure(figsize=(8, 5))
plt.bar(freq.index, freq.values, width=1.0, edgecolor='black')
plt.xlabel("Duration (T)")
plt.ylabel("Frequency")
plt.title("Distribution of Survival Times (T)")
plt.show()

### b. Unedited full pipeline

In [ ]:
if False:
    import numpy as np
    from sklearn.metrics import f1_score
    from sklearn.feature_selection import SelectFromModel
    from sklearn.ensemble import RandomForestClassifier
    from sksurv.linear_model import CoxnetSurvivalAnalysis
    from sksurv.metrics import concordance_index_censored

    # ============================================
    # STEP 0 — Prepare Data
    # ============================================
    # X: feature matrix (DataFrame)
    # T: time-to-event (array)
    # E: event indicator (array, 1=event occurred, 0=censored)
    # Y: classification labels (array)

    drop_columns = ['school_year', 'student_id_pseudonimized', 'category_name',
        'gross_receivables', 'amount_discounted', 'adjustments', 'due_date',
        'due_date_prev_1', 'due_date_prev_2', 'date_fully_paid', 'last_payment_date',
        'plan_type_Plan - D', 'plan_type_Plan - E', 'plan_type_nan', 'dtp_bracket']

    drop_columns = drop_columns + ['plan_type_Plan - A', 'plan_type_Plan - B', 'plan_type_Plan - C']

    df_data_surv = df_credit_sales.drop(columns=drop_columns)
    df_data_surv.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'], inplace=True)
    df_data_surv


    X = df_data_surv.drop(columns=['days_elapsed_until_fully_paid', 'censor'])
    T = df_data_surv['days_elapsed_until_fully_paid']
    E = df_data_surv['censor']
    Y = df_data[args.target_feature]

    # Avoid negative values by shifting by the days of pre-paid period
    earliest_pre_payment = np.minimum(T, 0) # Maximum to only get pre-payments
    ε = 1e-6 # Used to avoid zero values
    T = T - earliest_pre_payment + ε



    # Convert to structured array for scikit-survival
    survival_data = np.array([(bool(e), t) for e, t in zip(E, T)],
                            dtype=[('event', 'bool'), ('time', 'float')])

    # ============================================
    # STEP 1 — Define Candidate Penalties
    # ============================================
    penalties = [1e-4, 1e-3, 1e-2, 1e-1, 1]

    best_penalty = None
    best_c_index = -np.inf

    # ============================================
    # STEP 2 — Tune Survival Model
    # ============================================
    for λ in penalties:
        model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[λ])
        model.fit(X, survival_data)
        
        risk_scores = model.predict(X)
        c_index = concordance_index_censored(E.astype(bool), T, risk_scores)[0]
        
        if c_index > best_c_index:
            best_c_index = c_index
            best_penalty = λ

    print("Best Penalty:", best_penalty, "Best C-index:", best_c_index)

    # ============================================
    # STEP 3 — Retrain Survival Model
    # ============================================
    survival_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[best_penalty])
    survival_model.fit(X, survival_data)

    # ============================================
    # STEP 4 — Generate Survival-Based Features
    # ============================================
    df_survival = generate_survival_features(X, T, E, X_train, X_test, best_penalty)

    # ============================================
    # STEP 5 — Train Initial Classifier
    # ============================================
    clf = RandomForestClassifier(random_state=42)
    clf.fit(df_survival, survival_train)

    # Evaluate initial classifier
    Y_pred_initial = clf.predict(df_survival)
    f1_initial = f1_score(Y, Y_pred_initial, average="macro")
    print("Initial F1 Score:", f1_initial)

    # ============================================
    # STEP 6 — Feature Selection
    # ============================================
    selector = SelectFromModel(clf, prefit=True)
    X_selected = selector.transform(df_survival)

    # ============================================
    # STEP 7 — Train Final Classifier
    # ============================================
    final_clf = RandomForestClassifier(random_state=42)
    final_clf.fit(X_selected, Y)

    # ============================================
    # STEP 8 — Predictions
    # ============================================
    Y_pred_final = final_clf.predict(X_selected)
    f1_final = f1_score(Y, Y_pred_final, average="macro")

    print("Final F1 Score:", f1_final)


## 2. Enhanced with Survival Features

### a. Calculating the best penalty (λ) 

In [ ]:
# Helper columns
drop_columns = ['school_year', 'student_id_pseudonimized', 'category_name',
       'gross_receivables', 'amount_discounted', 'adjustments', 'due_date',
       'due_date_prev_1', 'due_date_prev_2', 'date_fully_paid', 'last_payment_date']

# Plan type columns
drop_columns = drop_columns + ['plan_type_Plan - A', 'plan_type_Plan - B', 'plan_type_Plan - C',
                               'plan_type_Plan - D', 'plan_type_Plan - E', 'plan_type_nan']

drop_columns = drop_columns + [args.target_feature]

df_data_surv = df_credit_sales.drop(columns=drop_columns)
df_data_surv.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'], inplace=True)
df_data_surv = df_data_surv[df_data_surv['days_elapsed_until_fully_paid'] > 0]
df_data_surv

In [ ]:
from sklearn.preprocessing import LabelEncoder

target_feature = args.target_feature

# Extract variables for training the survival model
X = df_data_surv.drop(columns=['days_elapsed_until_fully_paid', 'censor'])
T = df_data_surv['days_elapsed_until_fully_paid']
E = df_data_surv['censor']

# Avoid negative values by shifting by the days of pre-paid period
earliest_pre_payment = np.minimum(T, 0) # Maximum to only get pre-payments
ε = 1e-6 # Used to avoid zero values
T = T - earliest_pre_payment + ε

In [ ]:
import numpy as np
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

from machine_learning.utils.features.generate_survival_features import generate_survival_features

# ============================================
# STEP 0 — Initialize and Prepare Data
# ============================================

# Build survival array
survival_train = np.array(
    [(bool(e), float(t)) for e, t in zip(E.values, T.values)],
    dtype=[('event', 'bool'), ('time', 'float')]
)

# ============================================
# STEP 1 — Tune Survival Model
# ============================================
penalties = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]
best_penalty, best_c_index = None, -np.inf

for λ in penalties:
    model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[λ], normalize=True)
    model.fit(X, survival_train)
    risk_scores = model.predict(X)
    c_index = concordance_index_censored(E.astype(bool), T, risk_scores)[0]
    if c_index > best_c_index:
        best_penalty, best_c_index = λ, c_index

print(f"Best Penalty: {best_penalty} | Best C-index: {best_c_index}")

# ============================================
# STEP 2 — Retrain Survival Model
# ============================================
survival_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[best_penalty])
survival_model.fit(X, survival_train)

In [ ]:
best_params = {'penalizer': 0.1,
  'l1_ratio': 1,
  'baseline_estimation_method': 'breslow',
  'robust': True,
  'step_size': 0.5
}

### b. Hyperparameter tuning (various balancing strategies, parameters, and models)

In [ ]:
from machine_learning import (
    AdaBoostPipeline,
    DecisionTreePipeline,
    GaussianNaiveBayesPipeline,
    KNearestNeighborPipeline,
    RandomForestPipeline,
    XGBoostPipeline,
    StackedEnsemblePipeline,
    MultiLayerPerceptronPipeline,
    TransformerPipeline,
)

models = {
    "ada_boost": AdaBoostPipeline,
    "decision_tree": DecisionTreePipeline,
    "gaussian_naive_bayes": GaussianNaiveBayesPipeline,
    "knn": KNearestNeighborPipeline,
    "random_forest": RandomForestPipeline,
    "xgboost": XGBoostPipeline,
    #"stacked_ensemble": StackedEnsemblePipeline,
    "nn_mlp": MultiLayerPerceptronPipeline,
    #"nn_rnn": RecurrentNeuralNetworkPipeline,
    #"nn_transformer": TransformerPipeline
}

# Since these models are trained in the GPU, it's best
# to not to parallel compute to avoid bugs
do_not_parallel_compute = ['xg_boost', 'nn_mlp', 'nn_transformer']

In [ ]:
from machine_learning.utils.features.generate_thresholds import generate_thresholds

#balance_strategies = ["smote", "borderline_smote", "smoteenn", "smotetomek", "hybrid"]
balance_strategies = ["borderline_smote", "smote_enn", "smote_tomek"]

# thresholds used in the hybrid approach
thresholds = generate_thresholds(min_threshold=0.5, max_threshold=0.9, step=0.1)

In [ ]:
# To silence the error when running knn:
# UserWarning: Could not find the number of physical cores for the following reason:
# [WinError 2]
import os

os.environ['OMP_NUM_THREADS'] = '16'

In [ ]:
# Older method used to debug if there are any changes in the model's code
if False:
    from MachineLearning.Utils.run_models_single_threaded import run_survival_model_experiments

    df_results = run_survival_model_experiments(df_data_surv, models, balance_strategies, args, best_penalty, thresholds)

In [ ]:
# Helper columns
drop_columns = ['school_year', 'student_id_pseudonimized', 'category_name',
       'gross_receivables', 'amount_discounted', 'adjustments', 'due_date_prev_1',
       'due_date_prev_2', 'date_fully_paid', 'last_payment_date']

# Payment plans
#drop_columns = drop_columns + ['plan_type_Plan - A', 'plan_type_Plan - B', 'plan_type_Plan - C',
#                               'plan_type_Plan - D', 'plan_type_Plan - E', 'plan_type_nan']

# Survival related features
drop_columns = drop_columns + ['censor', 'days_elapsed_until_fully_paid']

# Only extract invoices with payments
df_data = df_credit_sales[df_credit_sales['censor'] == 1]

df_data = df_data.drop(columns=drop_columns)

# Drop invoices with missing critical features
df_data.dropna(subset=['dtp_1', 'dtp_2', 'dtp_3', 'dtp_4'], inplace=True)
df_data

In [ ]:
from machine_learning.models.stacked_ensemble import StackedEnsemblePipeline
from machine_learning.utils.features.generate_survival_features import generate_survival_features


args = {}

df_surv = df_data_surv.copy()

X = df_surv.drop(columns=['days_elapsed_until_fully_paid', 'censor'])
T = df_surv['days_elapsed_until_fully_paid']
E = df_surv['censor']

# Avoid negative values by shifting by the days of pre-paid period
earliest_payment = np.minimum(T, 0) # Maximum to only get pre-payments
ε = 1e-6 # Used to avoid zero values
T = T - earliest_payment + ε

X_surv_train, X_test_test = generate_survival_features(X, T, E, X_train, X_test, best_params, time_points=[16, 30, 58, 76, 118, 150, 284, 306])


estimators={
    "adaboost":      {"learning_rate": 0.1, "n_estimators": 10},
    "random_forest": {"max_depth": 10, "min_samples_leaf": 1, "n_estimators": 100},
}

sep = StackedEnsemblePipeline(X_surv_train, X_test_test, y_train, y_test, args)
sep.initialize_model(estimators=estimators).fit(use_feature_selection=False, threshold="median")

In [ ]:
result_baseline = sep.evaluate().show_results()

In [ ]:
sep.features.weights

In [ ]:
from machine_learning.utils.training.run_models_parallel import SurvivalExperimentRunner

# Create an experiment runner instance
runner = SurvivalExperimentRunner(
    df_data=df_data,
    df_data_surv=df_data_surv,
    models=models,
    balance_strategies=balance_strategies,
    args=args,
    best_parameters=best_params,
    thresholds=thresholds,
    n_jobs=-1,
    do_not_parallel_compute=do_not_parallel_compute,

    feature_selection_baseline=True,
    feature_selection_enhanced=True
)

# Run experiments
df_results = runner.run()

In [ ]:
df_results

In [ ]:
df_results.sort_values(by='enhanced_accuracy', ascending=False)

In [ ]:
df_results.sort_values(by='enhanced_precision_macro', ascending=False)

In [ ]:
df_results.sort_values(by='enhanced_f1_macro', ascending=False)

In [ ]:
df_results.sort_values(by='enhanced_roc_auc_macro', ascending=False)

In [ ]:
import pandas as pd

# File path
file_path = r"C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\MachineLearning\Results\06 model_results - no feature selection.xlsx"

# Read the Excel file
df = pd.read_excel(file_path)

# Choose which score column to use
score_column = "enhanced_roc_auc_macro"   # <-- change this to the metric you want (e.g., "f1", "roc_auc", etc.)

# Create a combined column: balance_strategy + threshold (only for hybrid)
df["balance_strategy_full"] = df.apply(
    lambda row: f"{row['balance_strategy']}_{row['undersample_threshold']}"
    if row["balance_strategy"] == "hybrid" else row["balance_strategy"],
    axis=1
)

# Compute mean score per model+parameters+strategy
grouped = (
    df.groupby(["model", "parameters", "balance_strategy_full"])[score_column]
    .mean()
    .reset_index()
)

# Rank strategies within each model+parameters group
grouped["rank"] = grouped.groupby(["model", "parameters"])[score_column] \
                         .rank(method="first", ascending=False)

# Assign weighted points: top 1 → 5, top 2 → 4, … top 5 → 1
def assign_points(rank):
    if rank == 1: return 5
    elif rank == 2: return 4
    elif rank == 3: return 3
    elif rank == 4: return 2
    elif rank == 5: return 1
    else: return 0

grouped["points"] = grouped["rank"].apply(assign_points)

# Aggregate across all models to see total points per strategy
strategy_scores = (
    grouped.groupby("balance_strategy_full")["points"]
    .sum()
    .reset_index()
    .sort_values("points", ascending=False)
)

# Show results
print("Weighted ranking per model+parameters:")
print(grouped.sort_values(["model", "parameters", "rank"]))

print("\nGlobal tally of weighted points per balance strategy:")
print(strategy_scores)

In [ ]:
import pandas as pd

# File path
file_path = r"C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\Results\08 - model_results - fixed enhancement features (with feature selection).xlsx"

# Read the Excel file
df = pd.read_excel(file_path)

# Choose which score column to use
score_column = "enhanced_f1_macro"   # <-- change this to the metric you want (e.g., "f1", "roc_auc", etc.)

# Create a combined column: balance_strategy + threshold (only for hybrid)
df["balance_strategy_full"] = df.apply(
    lambda row: f"{row['balance_strategy']}_{row['undersample_threshold']}"
    if row["balance_strategy"] == "hybrid" else row["balance_strategy"],
    axis=1
)

# Compute mean score per model+parameters+strategy
grouped = (
    df.groupby(["model", "parameters", "balance_strategy_full"])[score_column]
    .mean()
    .reset_index()
)

# Rank strategies within each model+parameters group
grouped["rank"] = grouped.groupby(["model", "parameters"])[score_column] \
                         .rank(method="first", ascending=False)

# Assign weighted points: top 1 → 5, top 2 → 4, … top 5 → 1
def assign_points(rank):
    if rank == 1: return 5
    elif rank == 2: return 4
    elif rank == 3: return 3
    elif rank == 4: return 2
    elif rank == 5: return 1
    else: return 0

grouped["points"] = grouped["rank"].apply(assign_points)

# Aggregate across all models to see total points per strategy
strategy_scores = (
    grouped.groupby("balance_strategy_full")["points"]
    .sum()
    .reset_index()
    .sort_values("points", ascending=False)
)

# Show results
print("Weighted ranking per model+parameters:")
print(grouped.sort_values(["model", "parameters", "rank"]))

print("\nGlobal tally of weighted points per balance strategy:")
print(strategy_scores)

In [ ]:
import pickle
import os

run_folder_path = r"C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\Results\2026_03_06_03"

with open(os.path.join(run_folder_path, "results.pkl"), "rb") as f:
        results_df = pickle.load(f)

results_df

In [ ]:
results_df.to_excel('Results/Test.xlsx', index=False)

In [ ]:
from machine_learning.utils.io.save_results_to_folder import load_training_results



path = r"C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\Results\2026_03_08_02"

data = load_training_results(path)

In [ ]:
data[2]

## c. Forecasting pipeline

In [ ]:
from feature_engineering.credit_sales_machine_learning import CreditSales

cs_test = CreditSales(df_revenues, df_enrollees, args,
                      drop_fully_paid_invoices=True,
                      drop_back_account_transactions=True,
                      calculate_payment_amounts=True,
                      add_description=True,
                      drop_missing_dtp=False)
df_cs_test = cs_test.show_data()
df_cs_test

In [5]:
from machine_learning.utils.io.load_results_from_folder import SessionStore

results_root = r"C:\Users\rjbel\Python\Notebooks\Mapua\Thesis\Results"
session = SessionStore(results_root)

In [8]:
results = session.load('2026_03_13_01')

In [11]:
results.keys()

dict_keys(['model_results_df', 'class_mappings', 'survival_results', 'metadata'])

In [13]:
results['metadata']

{'timestamp': '2026-03-13T03:08:32.371872',
 'num_models_trained': 7,
 'model_names': ['decision_tree',
  'random_forest',
  'xgboost',
  'ada_boost',
  'gaussian_naive_bayes',
  'knn',
  'nn_mlp'],
 'num_experiments': 300,
 'results_format': 'sqlite',
 'training_start_time': '2026-03-13T02:46:46.416490',
 'training_end_time': '2026-03-13T03:08:32.370869',
 'training_run_time': '0:21:45.954379',
 'run_folder_path': 'Results\\2026_03_13_01'}

In [14]:
results['survival_results']

{'best_c_index': 0.7696864930189864,
 'best_surv_parameters': {'alpha': 0.1, 'l1_ratio': 1.0},
 'best_time_points': [1, 16, 30, 58, 76, 118, 150, 284, 306]}